In [2]:
import pandas
import pathlib

In [5]:
files = list(pathlib.Path('.').glob('json/*.json'))

## Collect files

In [6]:
dfs = [pandas.read_json(file) for file in files]

In [7]:
df = pandas.concat(dfs, ignore_index=True)

In [12]:
df.head()

,year,nomination_url,award,url,actor,actor_url,film_title,film_url,character_name,citation,is_winner,has_acceptance_speech,acceptance_speech_text,acceptance_speech_url
0,1927/28 (1st),https://awardsdatabase.oscars.org/Search/Nomin...,ACTOR,https://awardsdatabase.oscars.org/Search/Nomin...,Richard Barthelmess,https://awardsdatabase.oscars.org/Search/Nomin...,The Noose,https://awardsdatabase.oscars.org/Search/Nomin...,"{""Nickie Elkins""};",None,False,False,NaN,NaN
1,1927/28 (1st),https://awardsdatabase.oscars.org/Search/Nomin...,ACTOR,https://awardsdatabase.oscars.org/Search/Nomin...,Emil Jannings,https://awardsdatabase.oscars.org/Search/Nomin...,The Last Command,https://awardsdatabase.oscars.org/Search/Nomin...,"{""General Dolgorucki [Grand Duke Sergius Alexa...",None,True,False,NaN,NaN
2,1927/28 (1st),https://awardsdatabase.oscars.org/Search/Nomin...,ACTRESS,https://awardsdatabase.oscars.org/Search/Nomin...,Louise Dresser,https://awardsdatabase.oscars.org/Search/Nomin...,A Ship Comes In,https://awardsdatabase.oscars.org/Search/Nomin...,"{""Mrs. Pleznik""}",None,False,False,NaN,NaN
3,1927/28 (1st),https://awardsdatabase.oscars.org/Search/Nomin...,ACTRESS,https://awardsdatabase.oscars.org/Search/Nomin...,Janet Gaynor,https://awardsdatabase.oscars.org/Search/Nomin...,7th Heaven,https://awardsdatabase.oscars.org/Search/Nomin...,"{""Diane""};",None,True,False,NaN,NaN
4,1927/28 (1st),https://awardsdatabase.oscars.org/Search/Nomin...,ACTRESS,https://awardsdatabase.oscars.org/Search/Nomin...,Gloria Swanson,https://awardsdatabase.oscars.org/Search/Nomin...,Sadie Thompson,https://awardsdatabase.oscars.org/Search/Nomin...,"{""Sadie Thompson""}",None,False,False,NaN,NaN


## Refactoring

In [13]:
df.award = df.award.map(lambda x: isinstance(x, str) and x.lower().title() or x)

The year comes like __2000 (xth edition)__ and we need to correct that

In [16]:
for item in df.itertuples():
    value = df.loc[item.Index, 'year']
    if value is not None:
        year, edition = value.split(' (')
        edition = edition.removesuffix(')').replace('th', '').replace('st', '').replace('nd', '').replace('rd', '')
        df.loc[item.Index, 'year'] = str(year)
        df.loc[item.Index, 'edition'] = int(edition)

In [ ]:
def fix_character(value: str | None) -> str | None:
    if value is None:
        return value
    return value.replace('{', '').replace('}', '').replace('"', '').strip()

In [ ]:
df.character_name = df.character_name.map(fix_character)

In [22]:
def fix_year(value: str | None) -> str | None:
    if value is None:
        return value
    
    if '/' in value:
        parts = value.split('/')
        year = int(parts[0]) + 1
        return year
    return int(value)

In [23]:
df.year = df.year.map(fix_year)

In [29]:
df.sort_values(by=['edition', 'award'], ascending=[False, True], inplace=True)

In [35]:
df = df[['year', 'award', 'actor', 'is_winner', 'character_name', 'actor_url', 'url', 'film_title','film_url','citation', 'nomination_url','has_acceptance_speech','acceptance_speech_text','acceptance_speech_url','edition']]

In [36]:
df.to_csv('oscars.csv', index=False)